In [1]:
# All package imports (run this cell first)
import sys
import json
from pathlib import Path

import torch
from datasets import load_from_disk
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer
from peft import PeftModel, get_peft_model, LoraConfig, TaskType

## Kernel check (use root .venv)

Run this cell first to confirm the notebook is using the project's root `.venv`.

In [2]:
# Kernel verification
_venv_ok = "My-Crew-Manager" in sys.executable and ".venv" in sys.executable
print(f"Python: {sys.executable}")

Python: c:\Users\Aaron\GitHub Repos\My-Crew-Manager\.venv\Scripts\python.exe


# Model Part 2 - Backlog Generation

From structured Part 1 JSON to backlog text (Epic to Sub-Epic to User Story to Task).

This notebook expects an existing model2_part1_to_backlog.jsonl dataset in llms/fine_tune/dataset.

## Setup paths

In [3]:
# Resolve AI root
_cwd = Path.cwd()
_ai_root = _cwd if (_cwd / "llms").exists() else (_cwd / "AI" if (_cwd / "AI").exists() else _cwd)
if str(_ai_root) not in sys.path:
    sys.path.insert(0, str(_ai_root))

FINE_TUNE_DIR = _ai_root / "llms" / "fine_tune"
DATASET_DIR = FINE_TUNE_DIR / "dataset"
TOKENIZED_DIR = FINE_TUNE_DIR / "tokenized"
OUTPUT_DIR = FINE_TUNE_DIR / "qwen_model2_backlog_lora_1p5b"

print(f"AI root: {_ai_root}")
print(f"Dataset: {DATASET_DIR}")

AI root: c:\Users\Aaron\GitHub Repos\My-Crew-Manager\AI
Dataset: c:\Users\Aaron\GitHub Repos\My-Crew-Manager\AI\llms\fine_tune\dataset


## Step 1: Import existing Model 2 dataset

Load the prepared `model2_part1_to_backlog.jsonl` dataset and preview sample records.

In [5]:
# Import and preview Model 2 dataset
dataset_file = DATASET_DIR / "model2_part1_to_backlog_epic_v1.jsonl"
if not dataset_file.exists():
    raise FileNotFoundError(
        f"Missing dataset: {dataset_file}. Generate or convert datasets first."
    )

rows = []
for line in dataset_file.read_text(encoding="utf-8").splitlines():
    line = line.strip()
    if not line:
        continue
    rows.append(json.loads(line))

print(f"Loaded {len(rows)} examples from {dataset_file.name}")
print("Model 2 contract:")
print("- Prompt: Model 1 response text (Title, Summary, Roles, Features, Goals, Timeline)")
print("- Response: Backlog only in Epic -> Sub-Epic -> User Story -> Task hierarchy")
print("- Structure rule: each Epic has exactly 1 Sub-Epic, 1 User Story, and 2 Tasks")
if rows:
    print("\nSample prompt snippet (Model 1 text):")
    print(rows[0]["prompt"][:280] + ("..." if len(rows[0]["prompt"]) > 280 else ""))
    print("\nSample backlog response snippet:")

Loaded 50 examples from model2_part1_to_backlog_epic_v1.jsonl
Model 2 contract:
- Prompt: Model 1 response text (Title, Summary, Roles, Features, Goals, Timeline)
- Response: Backlog only in Epic -> Sub-Epic -> User Story -> Task hierarchy
- Structure rule: each Epic has exactly 1 Sub-Epic, 1 User Story, and 2 Tasks

Sample prompt snippet (Model 1 text):
=== EventEase Smart Event Planner ===
Summary:
EventEase is a platform that helps users plan and manage events such as weddings, conferences, and parties. It provides AI-driven recommendations for venues, vendors, and schedules. The system aims to simplify event planning and redu...

Sample backlog response snippet:


In [6]:
import re

# Dataset cleanliness and contract audit (Epic hierarchy).
if "dataset_file" not in globals():
    dataset_file = DATASET_DIR / "model2_part1_to_backlog_epic_v1.jsonl"

if not dataset_file.exists():
    raise FileNotFoundError(f"Missing dataset file for audit: {dataset_file}")

def _audit_model2_response_structure(text: str) -> tuple[bool, str]:
    epics = []
    current_epic = None
    sub_count = story_count = task_count = 0

    for raw in text.splitlines():
        line = raw.strip()
        if not line:
            continue

        epic_match = re.match(r"(?i)^epic\s*(\d+)\s*:\s+.+$", line)
        if epic_match:
            if current_epic is not None and not (sub_count == 1 and story_count == 1 and task_count == 2):
                return False, (
                    f"Epic {current_epic} invalid cardinality "
                    f"(sub={sub_count}, story={story_count}, task={task_count})"
                )
            current_epic = int(epic_match.group(1))
            epics.append(current_epic)
            sub_count = story_count = task_count = 0
            continue

        # Accept flat and legacy decimal numbering for robustness.
        if re.match(r"(?i)^-?\s*sub-?epic\s*\d+(?:\.\d+)?\s*:\s+.+$", line):
            sub_count += 1
            continue
        if re.match(r"(?i)^-?\s*user\s*story\s*\d+(?:\.\d+)*\s*:\s+.+$", line):
            story_count += 1
            continue
        if re.match(r"(?i)^-?\s*task\s*\d+(?:\.\d+)*\s*:\s+.+$", line):
            task_count += 1
            continue

    if current_epic is not None and not (sub_count == 1 and story_count == 1 and task_count == 2):
        return False, (
            f"Epic {current_epic} invalid cardinality "
            f"(sub={sub_count}, story={story_count}, task={task_count})"
        )

    if not (4 <= len(epics) <= 6):
        return False, f"Epic count out of range: {len(epics)}"

    forbidden = re.search(
        r"(?im)^\s*(Status|Summary|Roles|Features|Timeline|Proposal\s*/\s*Input)\s*:",
        text,
    )
    if forbidden:
        return False, f"Forbidden section found: {forbidden.group(1)}"

    return True, "ok"

total_rows = 0
valid_rows = 0
bad_rows = []

for idx, raw in enumerate(dataset_file.read_text(encoding="utf-8").splitlines(), start=1):
    line = raw.strip()
    if not line:
        continue
    total_rows += 1
    try:
        row = json.loads(line)
    except json.JSONDecodeError as exc:
        bad_rows.append((idx, f"invalid_json: {exc}"))
        continue

    prompt = (row.get("prompt") or "").strip()
    response = (row.get("response") or "").strip()
    if not prompt or not response:
        bad_rows.append((idx, "missing_prompt_or_response"))
        continue

    ok, reason = _audit_model2_response_structure(response)
    if ok:
        valid_rows += 1
    else:
        bad_rows.append((idx, reason))

print("MODEL 2 DATASET CLEANLINESS REPORT")
print(f"Dataset: {dataset_file}")
print(f"Rows checked: {total_rows}")
print(f"Valid rows: {valid_rows}")
print(f"Invalid rows: {len(bad_rows)}")

if bad_rows:
    print("\nFirst 10 invalid rows:")
    for row_idx, reason in bad_rows[:10]:
        print(f"- row {row_idx}: {reason}")
    raise RuntimeError("Dataset contract violations found. Fix dataset before training.")
else:
    print("Dataset contract check passed.")

MODEL 2 DATASET CLEANLINESS REPORT
Dataset: c:\Users\Aaron\GitHub Repos\My-Crew-Manager\AI\llms\fine_tune\dataset\model2_part1_to_backlog_epic_v1.jsonl
Rows checked: 50
Valid rows: 50
Invalid rows: 0
Dataset contract check passed.


## Step 2: Prepare tokenized dataset

In [7]:
from llms.fine_tune.prepare_dataset import prepare_model2_dataset

DATASET_FILENAME = "model2_part1_to_backlog_epic_v1.jsonl"
source_dataset_file = DATASET_DIR / DATASET_FILENAME
tokenized_path = TOKENIZED_DIR / "tokenized_model2_qwen_epic_dualprompt_v3"
MAX_LENGTH = 768
FORCE_REBUILD_TOKENIZED = False

if not source_dataset_file.exists():
    raise FileNotFoundError(f"Missing source dataset: {source_dataset_file}")

source_count = sum(1 for line in source_dataset_file.read_text(encoding="utf-8").splitlines() if line.strip())
print(f"Source examples in JSONL ({DATASET_FILENAME}): {source_count}")

rebuild_needed = FORCE_REBUILD_TOKENIZED or not tokenized_path.exists()
if tokenized_path.exists() and not rebuild_needed:
    dataset = load_from_disk(str(tokenized_path))
    tokenized_count = len(dataset)
    if tokenized_count != source_count:
        print(
            f"Tokenized dataset is stale (tokenized={tokenized_count}, source={source_count}). "
            f"Rebuilding..."
        )
        rebuild_needed = True

if rebuild_needed:
    if tokenized_path.exists():
        import shutil
        shutil.rmtree(tokenized_path)
    dataset = prepare_model2_dataset(
        model_name="qwen",
        max_length=MAX_LENGTH,
        dataset_filename=DATASET_FILENAME,
        output_dir=str(tokenized_path),
    )
    print(f"Rebuilt tokenized dataset at {tokenized_path}")
else:
    print(f"Loaded tokenized dataset from {tokenized_path}")

# 60/20/20 split: train / eval / holdout
split_primary = dataset.train_test_split(test_size=0.4, seed=42)
train_dataset = split_primary["train"]

split_secondary = split_primary["test"].train_test_split(test_size=0.5, seed=42)
eval_dataset = split_secondary["train"]
holdout_dataset = split_secondary["test"]

print(f"Total tokenized examples: {len(dataset)}")
print(f"Train size (60%): {len(train_dataset)}")
print(f"Eval size (20%): {len(eval_dataset)}")
print(f"Holdout size (20%): {len(holdout_dataset)}")

Source examples in JSONL (model2_part1_to_backlog_epic_v1.jsonl): 50
Loaded tokenized dataset from c:\Users\Aaron\GitHub Repos\My-Crew-Manager\AI\llms\fine_tune\tokenized\tokenized_model2_qwen_epic_dualprompt_v3
Total tokenized examples: 50
Train size (60%): 30
Eval size (20%): 10
Holdout size (20%): 10


## Step 3: Load model & apply LoRA

In [4]:
MODEL_ID = "Qwen/Qwen2-1.5B-Instruct"
BATCH_SIZE = 2
EPOCHS = 5

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,
    trust_remote_code=True,
)
if torch.cuda.is_available():
    model = model.to("cuda")

peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=32,
    lora_dropout=0.1,
    bias="none",
)
model = get_peft_model(model, peft_config)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

## Step 4: Train iteratively until target

Train in capped rounds, evaluate after each round, and stop early when quality targets are reached.

In [12]:
import gc
import inspect
import math
import os
import random
import shutil
from pathlib import Path

os.environ.setdefault("TENSORBOARD_LOGGING_DIR", str(OUTPUT_DIR / "logs"))
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

# Backlog-tuned defaults (stricter than Model 1, still configurable).
TARGET_EVAL_LOSS = 1.10
TARGET_PERPLEXITY = 3.00
MAX_ROUNDS = 10
MIN_ROUNDS = 1
PATIENCE_ROUNDS = 2
EPOCHS_PER_ROUND = 3
SEEDS = [42, 123]

# Memory-safe effective batch size: 2 (1 x accumulation 2).
TRAIN_BATCH_SIZE = 1
GRAD_ACC_STEPS = 2

TRIALS_DIR = OUTPUT_DIR / "trials"
TRIALS_DIR.mkdir(parents=True, exist_ok=True)


def set_global_seed(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def clear_cuda_cache() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


training_kwargs_base = {
    "per_device_train_batch_size": TRAIN_BATCH_SIZE,
    "gradient_accumulation_steps": GRAD_ACC_STEPS,
    "per_device_eval_batch_size": 1,
    "num_train_epochs": EPOCHS_PER_ROUND,
    "logging_steps": 10,
    "fp16": torch.cuda.is_available(),
    "report_to": "none",
    "save_strategy": "no",
    "dataloader_pin_memory": False,
    "prediction_loss_only": True,
    "torch_empty_cache_steps": 1,
    "disable_tqdm": True,
}
if "evaluation_strategy" in inspect.signature(TrainingArguments.__init__).parameters:
    training_kwargs_base["evaluation_strategy"] = "no"
else:
    training_kwargs_base["eval_strategy"] = "no"

trial_summaries = []
global_best = None

for seed in SEEDS:
    print(f"\n{'=' * 80}")
    print(f"Starting trial for seed={seed}")
    print(f"{'=' * 80}")

    set_global_seed(seed)
    clear_cuda_cache()

    trial_dir = TRIALS_DIR / f"seed_{seed}"
    if trial_dir.exists():
        shutil.rmtree(trial_dir)
    trial_dir.mkdir(parents=True, exist_ok=True)

    trial_model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        dtype=torch.float16,
        trust_remote_code=True,
    )
    if torch.cuda.is_available():
        trial_model = trial_model.to("cuda")

    trial_model = get_peft_model(trial_model, peft_config)

    trial_kwargs = dict(training_kwargs_base)
    trial_kwargs["output_dir"] = str(trial_dir)
    trial_kwargs["seed"] = seed
    trial_kwargs["data_seed"] = seed

    trial_args = TrainingArguments(**trial_kwargs)

    trainer = Trainer(
        model=trial_model,
        args=trial_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        processing_class=tokenizer,
    )

    round_history = []
    best_round = None
    best_checkpoint_dir = None
    rounds_without_improvement = 0
    stop_reason = "max_rounds_reached"

    for round_num in range(1, MAX_ROUNDS + 1):
        print(f"\n--- Seed {seed} | Round {round_num}/{MAX_ROUNDS} ---")
        train_result = trainer.train()

        pred_output = trainer.predict(eval_dataset, metric_key_prefix="eval")
        eval_metrics = pred_output.metrics
        eval_loss = eval_metrics.get("eval_loss") or eval_metrics.get("test_loss")
        perplexity = math.exp(eval_loss) if eval_loss is not None and eval_loss < 20 else float("inf")

        round_checkpoint_dir = trial_dir / f"round_{round_num}"
        round_checkpoint_dir.mkdir(parents=True, exist_ok=True)
        trainer.save_model(str(round_checkpoint_dir))

        round_entry = {
            "seed": seed,
            "round": round_num,
            "train_loss": float(train_result.training_loss),
            "eval_loss": float(eval_loss) if eval_loss is not None else None,
            "perplexity": float(perplexity),
            "checkpoint_dir": str(round_checkpoint_dir),
        }
        round_history.append(round_entry)

        print(f"Train loss: {round_entry['train_loss']:.4f}")
        if eval_loss is not None:
            print(f"Eval loss: {eval_loss:.4f}")
        else:
            print("Eval loss: unavailable")
        print(f"Perplexity: {perplexity:.4f}" if math.isfinite(perplexity) else "Perplexity: inf")

        is_better = False
        if eval_loss is not None:
            if best_round is None:
                is_better = True
            else:
                if eval_loss < best_round["eval_loss"]:
                    is_better = True
                elif eval_loss == best_round["eval_loss"] and perplexity < best_round["perplexity"]:
                    is_better = True

        if is_better:
            best_round = round_entry
            best_checkpoint_dir = round_checkpoint_dir
            rounds_without_improvement = 0
            print("Improvement detected: updated best checkpoint.")
        else:
            rounds_without_improvement += 1
            print(f"No improvement. Patience counter: {rounds_without_improvement}/{PATIENCE_ROUNDS}")

        if eval_loss is not None and (eval_loss <= TARGET_EVAL_LOSS or perplexity <= TARGET_PERPLEXITY):
            if round_num >= MIN_ROUNDS:
                stop_reason = "target_reached"
                print(
                    f"Stopping: target reached at round {round_num} "
                    f"(eval_loss={eval_loss:.4f}, perplexity={perplexity:.4f})"
                )
                break

        if round_num >= MIN_ROUNDS and rounds_without_improvement >= PATIENCE_ROUNDS:
            stop_reason = "patience_exhausted"
            print(f"Stopping: no improvement for {PATIENCE_ROUNDS} rounds.")
            break

        clear_cuda_cache()

    if best_checkpoint_dir is None:
        print(f"Seed {seed}: no valid checkpoint selected; skipping holdout evaluation.")
        del trainer, trial_model
        clear_cuda_cache()
        continue

    holdout_base = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        dtype=torch.float16,
        trust_remote_code=True,
    )
    best_model = PeftModel.from_pretrained(holdout_base, str(best_checkpoint_dir))
    if torch.cuda.is_available():
        best_model = best_model.to("cuda")

    holdout_args = TrainingArguments(
        output_dir=str(trial_dir / "holdout_eval"),
        report_to="none",
        per_device_eval_batch_size=1,
        dataloader_pin_memory=False,
        prediction_loss_only=True,
        disable_tqdm=True,
    )
    holdout_trainer = Trainer(
        model=best_model,
        args=holdout_args,
        eval_dataset=holdout_dataset,
        processing_class=tokenizer,
    )

    holdout_pred = holdout_trainer.predict(holdout_dataset, metric_key_prefix="holdout")
    holdout_metrics = holdout_pred.metrics
    holdout_loss = holdout_metrics.get("holdout_loss") or holdout_metrics.get("test_loss")
    holdout_perplexity = math.exp(holdout_loss) if holdout_loss is not None and holdout_loss < 20 else float("inf")

    trial_summary = {
        "seed": seed,
        "stop_reason": stop_reason,
        "rounds_completed": len(round_history),
        "best_round": best_round,
        "best_checkpoint_dir": str(best_checkpoint_dir),
        "holdout_loss": float(holdout_loss) if holdout_loss is not None else None,
        "holdout_perplexity": float(holdout_perplexity),
    }
    trial_summaries.append(trial_summary)

    print(f"\nSeed {seed} summary")
    print(f"- Stop reason: {stop_reason}")
    print(f"- Best round: {best_round['round']} (eval_loss={best_round['eval_loss']:.4f}, perplexity={best_round['perplexity']:.4f})")
    print(
        f"- Holdout: loss={trial_summary['holdout_loss']:.4f}, "
        f"perplexity={trial_summary['holdout_perplexity']:.4f}"
        if trial_summary["holdout_loss"] is not None
        else "- Holdout: unavailable"
    )

    if global_best is None:
        global_best = trial_summary
    else:
        cur_loss = trial_summary["holdout_loss"]
        best_loss = global_best["holdout_loss"]
        cur_ppl = trial_summary["holdout_perplexity"]
        best_ppl = global_best["holdout_perplexity"]

        if cur_loss is not None and (best_loss is None or cur_loss < best_loss):
            global_best = trial_summary
        elif cur_loss is not None and best_loss is not None and cur_loss == best_loss and cur_ppl < best_ppl:
            global_best = trial_summary

    del holdout_trainer, best_model, holdout_base, trainer, trial_model
    clear_cuda_cache()

if not trial_summaries:
    raise RuntimeError("No successful trial completed. Cannot promote best model.")

print(f"\n{'=' * 80}")
print("TRIAL RESULTS")
print(f"{'=' * 80}")
for t in trial_summaries:
    br = t["best_round"]
    holdout_loss_txt = f"{t['holdout_loss']:.4f}" if t["holdout_loss"] is not None else "NA"
    print(
        f"Seed {t['seed']}: stop={t['stop_reason']}, rounds={t['rounds_completed']}, "
        f"best_round={br['round']} (eval_loss={br['eval_loss']:.4f}, perplexity={br['perplexity']:.4f}), "
        f"holdout_loss={holdout_loss_txt}, holdout_perplexity={t['holdout_perplexity']:.4f}"
    )

print(f"\nGlobal winner seed: {global_best['seed']}")
print(f"Winner checkpoint: {global_best['best_checkpoint_dir']}")

winner_seed = global_best["seed"]
winner_checkpoint_dir = Path(global_best["best_checkpoint_dir"])
winner_holdout_loss = global_best["holdout_loss"]
winner_holdout_perplexity = global_best["holdout_perplexity"]


Starting trial for seed=42


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.



--- Seed 42 | Round 1/10 ---
{'loss': '1.073', 'grad_norm': '0.4396', 'learning_rate': '4e-05', 'epoch': '0.6667'}
{'loss': '1.016', 'grad_norm': '0.4491', 'learning_rate': '2.889e-05', 'epoch': '1.333'}
{'loss': '0.9283', 'grad_norm': '0.3969', 'learning_rate': '1.778e-05', 'epoch': '2'}
{'loss': '0.8833', 'grad_norm': '0.3919', 'learning_rate': '6.667e-06', 'epoch': '2.667'}
{'train_runtime': '616.4', 'train_samples_per_second': '0.146', 'train_steps_per_second': '0.073', 'train_loss': '0.9684', 'epoch': '3'}
Train loss: 0.9684
Eval loss: 0.8358
Perplexity: 2.3067
Improvement detected: updated best checkpoint.
Stopping: target reached at round 1 (eval_loss=0.8358, perplexity=2.3067)


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.



--- Seed 123 | Round 1/10 ---
{'loss': '1.076', 'grad_norm': '0.3935', 'learning_rate': '4e-05', 'epoch': '0.6667'}
{'loss': '1.004', 'grad_norm': '0.4374', 'learning_rate': '2.889e-05', 'epoch': '1.333'}
{'loss': '0.9409', 'grad_norm': '0.4093', 'learning_rate': '1.778e-05', 'epoch': '2'}
{'loss': '0.8888', 'grad_norm': '0.4108', 'learning_rate': '6.667e-06', 'epoch': '2.667'}
{'train_runtime': '674.8', 'train_samples_per_second': '0.133', 'train_steps_per_second': '0.067', 'train_loss': '0.9697', 'epoch': '3'}
Train loss: 0.9697
Eval loss: 0.8360
Perplexity: 2.3072
Improvement detected: updated best checkpoint.
Stopping: target reached at round 1 (eval_loss=0.8360, perplexity=2.3072)


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]


Seed 123 summary
- Stop reason: target_reached
- Best round: 1 (eval_loss=0.8360, perplexity=2.3072)
- Holdout: loss=0.8347, perplexity=2.3041

TRIAL RESULTS
Seed 42: stop=target_reached, rounds=1, best_round=1 (eval_loss=0.8358, perplexity=2.3067), holdout_loss=0.8350, holdout_perplexity=2.3048
Seed 123: stop=target_reached, rounds=1, best_round=1 (eval_loss=0.8360, perplexity=2.3072), holdout_loss=0.8347, holdout_perplexity=2.3041

Global winner seed: 123
Winner checkpoint: c:\Users\Aaron\GitHub Repos\My-Crew-Manager\AI\llms\fine_tune\qwen_model2_backlog_lora_1p5b\trials\seed_123\round_1


## Step 4.1: Evaluate model performance

Compute evaluation loss and perplexity on the held-out evaluation split.

In [18]:
import gc
import math

# Evaluate promoted winner checkpoint on eval and holdout splits.
if "winner_checkpoint_dir" not in globals():
    winner_checkpoint_dir = OUTPUT_DIR
    winner_seed = "existing-adapter"
    print("Training cell skipped: evaluating existing adapter in OUTPUT_DIR.")

winner_checkpoint_dir = Path(winner_checkpoint_dir)
if not winner_checkpoint_dir.exists():
    raise RuntimeError(f"Winner checkpoint not found: {winner_checkpoint_dir}")

if torch.cuda.is_available():
    torch.cuda.empty_cache()

eval_base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,
    trust_remote_code=True,
)
winner_model = PeftModel.from_pretrained(eval_base, str(winner_checkpoint_dir))
if torch.cuda.is_available():
    winner_model = winner_model.to("cuda")

eval_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / "winner_eval"),
    report_to="none",
    per_device_eval_batch_size=1,
    dataloader_pin_memory=False,
    prediction_loss_only=True,
    disable_tqdm=True,
)
eval_trainer = Trainer(
    model=winner_model,
    args=eval_args,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
)

eval_metrics = eval_trainer.predict(eval_dataset, metric_key_prefix="eval").metrics
eval_loss = eval_metrics.get("eval_loss") or eval_metrics.get("test_loss")

holdout_metrics = eval_trainer.predict(holdout_dataset, metric_key_prefix="holdout").metrics
holdout_loss = holdout_metrics.get("holdout_loss") or holdout_metrics.get("test_loss")

eval_perplexity = math.exp(eval_loss) if eval_loss is not None and eval_loss < 20 else float("inf")
holdout_perplexity = math.exp(holdout_loss) if holdout_loss is not None and holdout_loss < 20 else float("inf")

print("Winner checkpoint evaluation")
print(f"Winner seed: {winner_seed}")
print(f"Winner checkpoint: {winner_checkpoint_dir}")
print(f"Eval loss: {eval_loss:.4f}" if eval_loss is not None else "Eval loss: unavailable")
print(f"Eval perplexity: {eval_perplexity:.4f}" if math.isfinite(eval_perplexity) else "Eval perplexity: inf")
print(f"Holdout loss: {holdout_loss:.4f}" if holdout_loss is not None else "Holdout loss: unavailable")
print(f"Holdout perplexity: {holdout_perplexity:.4f}" if math.isfinite(holdout_perplexity) else "Holdout perplexity: inf")

print("\nAll eval metrics:")
print(eval_metrics)
print("\nAll holdout metrics:")
print(holdout_metrics)

# Release eval objects before adapter save and inference.
for _name in ("eval_trainer", "winner_model", "eval_base"):
    if _name in globals():
        del globals()[_name]
if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

KeyboardInterrupt: 

## Step 5: Save adapter

In [ ]:
import gc
import shutil

# Defensive cleanup for out-of-order reruns before copying artifacts.
for _name in (
    "eval_trainer",
    "winner_model",
    "eval_base",
    "holdout_trainer",
    "best_model",
    "holdout_base",
    "model_infer",
    "base_model",
):
    if _name in globals():
        del globals()[_name]
if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if "winner_checkpoint_dir" not in globals():
    winner_checkpoint_dir = OUTPUT_DIR

winner_checkpoint_dir = Path(winner_checkpoint_dir)
if not winner_checkpoint_dir.exists():
    raise RuntimeError(f"Winner checkpoint missing: {winner_checkpoint_dir}")

# If training was skipped and winner points to OUTPUT_DIR, keep current adapter as-is.
if winner_checkpoint_dir.resolve() == OUTPUT_DIR.resolve():
    tokenizer.save_pretrained(str(OUTPUT_DIR))
    print(f"Training skipped; using existing adapter at {OUTPUT_DIR}")
    print("Set PEFT_ADAPTER_PATH_BACKLOG in AI/.env to use this adapter:")
else:
    # Clean existing adapter artifacts (keep trials folder if present).
    for child in OUTPUT_DIR.iterdir():
        if child.name == "trials":
            continue
        if child.is_dir():
            shutil.rmtree(child)
        else:
            child.unlink()

    # Copy winner adapter into final output directory.
    for child in winner_checkpoint_dir.iterdir():
        dst = OUTPUT_DIR / child.name
        if child.is_dir():
            shutil.copytree(child, dst)
        else:
            shutil.copy2(child, dst)

    # Ensure tokenizer files are present for inference.
    tokenizer.save_pretrained(str(OUTPUT_DIR))

    print(f"Promoted winner checkpoint from: {winner_checkpoint_dir}")
    print(f"Saved final adapter and tokenizer to {OUTPUT_DIR}")
    print("Set PEFT_ADAPTER_PATH_BACKLOG in AI/.env to use this adapter:")

Promoted winner checkpoint from: c:\Users\Aaron\GitHub Repos\My-Crew-Manager\AI\llms\fine_tune\qwen_model2_backlog_lora_1p5b\trials\seed_123\round_1
Saved final adapter and tokenizer to c:\Users\Aaron\GitHub Repos\My-Crew-Manager\AI\llms\fine_tune\qwen_model2_backlog_lora_1p5b
Set PEFT_ADAPTER_PATH_BACKLOG in AI/.env to use this adapter:
  PEFT_ADAPTER_PATH_BACKLOG=llms/fine_tune/qwen_model2_backlog_lora_1p5b


## Step 6: Quick inference test

In [6]:
import gc
import json
import os
import re
from datetime import datetime

# Final VRAM cleanup before loading inference models.
for _name in ("eval_trainer", "winner_model", "eval_base", "model", "trainer"):
    if _name in globals():
        del globals()[_name]
if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()

adapter_rel = os.getenv("PEFT_ADAPTER_PATH_BACKLOG", "llms/fine_tune/qwen_model2_backlog_lora_1p5b")
adapter_path_obj = Path(adapter_rel)
if adapter_path_obj.is_absolute():
    PROMOTED_ADAPTER_DIR = adapter_path_obj
else:
    PROMOTED_ADAPTER_DIR = (_ai_root / adapter_path_obj).resolve()

if not PROMOTED_ADAPTER_DIR.exists():
    raise FileNotFoundError(f"Promoted adapter not found: {PROMOTED_ADAPTER_DIR}")

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,
    trust_remote_code=True,
)
model_infer = PeftModel.from_pretrained(base_model, str(PROMOTED_ADAPTER_DIR))
if torch.cuda.is_available():
    model_infer = model_infer.to("cuda")

model_infer.eval()
for param in model_infer.parameters():
    param.requires_grad = False

if "DATASET_FILENAME" not in globals():
    DATASET_FILENAME = "model2_part1_to_backlog_epic_v1.jsonl"
dataset_file = DATASET_DIR / DATASET_FILENAME
if not dataset_file.exists():
    raise FileNotFoundError(f"Missing dataset for inference tests: {dataset_file}")

# Few-shot context disabled to reduce prompt noise and improve structural focus.
few_shot_examples = []
FEWSHOT_CONTEXT = ""

BACKLOG_PROMPT_PATH = _ai_root / "llms" / "prompts" / "backlog_prompt.txt"
OUTPUT_END_MARKER = "END_OF_BACKLOG"
USE_STRICT_PROMPT_RULES = True
MAX_RULE_LINES = 7
EXTRA_STRICT_RULE_LINES = [
    "Output only flat backlog hierarchy.",
    "Map Goals to epics in exact order: Goal 1->Epic 1, Goal 2->Epic 2, etc.",
    "Use exact headers only: Epic N, -Sub-Epic 1, -User Story 1, -Task 1, -Task 2.",
    "Start with Epic 1 and continue sequentially with no skipped epic numbers.",
    "Exactly one Sub-Epic and one User Story per Epic; exactly two Tasks per User Story.",
    "No Status/Summary/Roles/Features/Timeline/Proposal sections.",
    f"After final Task 2, output {OUTPUT_END_MARKER} and stop immediately.",
]


def _extract_backlog_rules(prompt_text: str) -> list[str]:
    rules = []
    in_rules_section = False
    for raw in prompt_text.splitlines():
        line = raw.strip()
        if not line:
            continue
        low = line.lower()

        if low.startswith("instructions:") or low.startswith("additional guidance:"):
            in_rules_section = True
            continue
        if low.startswith("formatting:") or low.startswith("context:"):
            continue

        if not in_rules_section:
            continue

        if line.startswith("-"):
            candidate = line.lstrip("-").strip()
            if candidate and candidate not in rules:
                rules.append(candidate)

    return rules


strict_rule_lines = []
if USE_STRICT_PROMPT_RULES and BACKLOG_PROMPT_PATH.exists():
    strict_prompt_text = BACKLOG_PROMPT_PATH.read_text(encoding="utf-8")
    strict_rule_lines.extend(_extract_backlog_rules(strict_prompt_text))

for extra in EXTRA_STRICT_RULE_LINES:
    if extra not in strict_rule_lines:
        strict_rule_lines.append(extra)
strict_rule_lines = strict_rule_lines[:MAX_RULE_LINES]

strict_prompt_integration_active = bool(USE_STRICT_PROMPT_RULES and strict_rule_lines)

BASE_GUIDED_PREFIX = (
    "Generate backlog only in strict flat hierarchy.\n"
    "Goals determine epic count exactly (N goals -> N epics).\n"
    "Use no decimals. Start with Epic 1 and continue sequentially.\n"
    "Per Epic: exactly 1 Sub-Epic 1, 1 User Story 1, and Task 1 + Task 2.\n"
    "Each task must be implementation-ready with concrete deliverable language.\n\n"
    "Template:\n"
    "Epic 1: <Goal 1 title>\n"
    "-Sub-Epic 1: <phase>\n"
    " -User Story 1: As a <role>, I want <capability> so that <benefit>\n"
    "  -Task 1: <specific technical action + artifact>\n"
    "  -Task 2: <specific technical action + artifact>\n\n"
    f"After final Task 2, output {OUTPUT_END_MARKER} on its own line and stop.\n"
)

if strict_prompt_integration_active:
    strict_constraints_block = "Strict rules:\n" + "\n".join(
        f"- {rule}" for rule in strict_rule_lines
    ) + "\n\n"
    GUIDED_PREFIX = BASE_GUIDED_PREFIX + strict_constraints_block
else:
    GUIDED_PREFIX = BASE_GUIDED_PREFIX

# Inference speed controls (increased token budget to 520 for full hierarchy generation)
QUICK_RUN_MODE = True
STRATEGIES_TO_RUN = ["B"] if QUICK_RUN_MODE else ["B", "C"]
TOKENS_BY_STRATEGY = {"B": 420, "C": 520}
SHOW_FULL_OUTPUT = True
EXPORT_OUTPUT_TXT = True
EXPORT_DIR = PROMOTED_ADAPTER_DIR / "inference_test_exports"

# Deterministic proposal selection (1-based dataset row numbers).
SELECTED_PROPOSAL_ROWS = [7]

if len(SELECTED_PROPOSAL_ROWS) < 1:
    raise ValueError(
        "SELECTED_PROPOSAL_ROWS must contain at least 1 row number "
        f"(received {len(SELECTED_PROPOSAL_ROWS)})."
    )
if not all(isinstance(x, int) for x in SELECTED_PROPOSAL_ROWS):
    raise ValueError("SELECTED_PROPOSAL_ROWS must contain integers only.")
if len(set(SELECTED_PROPOSAL_ROWS)) != len(SELECTED_PROPOSAL_ROWS):
    raise ValueError("SELECTED_PROPOSAL_ROWS must not contain duplicates.")
if min(SELECTED_PROPOSAL_ROWS) < 1:
    raise ValueError("SELECTED_PROPOSAL_ROWS must use 1-based row numbers (minimum is 1).")

selected_rows_sorted = sorted(SELECTED_PROPOSAL_ROWS)
selected_row_set = set(selected_rows_sorted)

_BAD_PHRASES = [
    "Instruction:",
    "Input:",
    "Solution:",
    "```",
    "Please continue from where you left off.",
    "Continue the tutorial",
]
_BAD_IDS = [tokenizer(x, add_special_tokens=False).input_ids for x in _BAD_PHRASES]
_BAD_IDS = [x for x in _BAD_IDS if x]


def _normalize_backlog_labels(text: str) -> str:
    normalized = text.split(OUTPUT_END_MARKER, 1)[0]
    normalized = re.sub(r"(?im)^\s*Backlog\s*:\s*$", "", normalized)
    normalized = re.sub(r"(?im)^\s*Goal\s*(\d+)\s*:\s*", r"Epic \1: ", normalized)
    # Keep flat Sub-Epic format (1, not 1.1)
    normalized = re.sub(r"(?im)^\s*Sub[- ]?Epic\s*(\d+)\s*:\s*", r"-Sub-Epic \1: ", normalized)
    # Keep flat User Story format (1, not 1.1.1)
    normalized = re.sub(r"(?im)^\s*User\s+Story\s+(\d+)\s*:\s*", r"-User Story \1: ", normalized)
    normalized = re.sub(r"(?im)^\s*Tasks\s*:\s*", "", normalized)
    # Keep flat Task format (1, 2, not 1.1.1.1)
    normalized = re.sub(r"(?im)^\s*Task\s+(\d+)\s*:\s*", r"-Task \1: ", normalized)
    return normalized.strip()


def _parse_backlog_hierarchy(text: str) -> list[dict]:
    """Parse flat backlog hierarchy (Epic 1, Sub-Epic 1, User Story 1, Task 1-2, no decimals)."""
    epics = []
    current_epic = None
    current_sub_epic = None
    current_story = None

    for raw in text.splitlines():
        line = raw.rstrip()
        stripped = line.strip()
        if not stripped:
            continue

        # Match "Epic N:" (flat, no decimals)
        epic_match = re.match(r"^Epic\s*(\d+)\s*:\s*(.+)$", stripped, re.IGNORECASE)
        if epic_match:
            current_epic = {
                "idx": int(epic_match.group(1)),
                "title": epic_match.group(2).strip(),
                "sub_epics": [],
            }
            epics.append(current_epic)
            current_sub_epic = None
            current_story = None
            continue

        # Match "Sub-Epic N:" (flat, no decimals like 1.1)
        sub_match = re.match(r"^-?\s*Sub-?Epic\s*(\d+)\s*:\s*(.+)$", stripped, re.IGNORECASE)
        if sub_match and current_epic is not None:
            current_sub_epic = {
                "idx": int(sub_match.group(1)),
                "title": sub_match.group(2).strip(),
                "stories": [],
            }
            current_epic["sub_epics"].append(current_sub_epic)
            current_story = None
            continue

        # Match "User Story N:" (flat, no decimals like 1.1.1)
        story_match = re.match(r"^-?\s*User\s+Story\s+(\d+)\s*:\s*(.+)$", stripped, re.IGNORECASE)
        if story_match and current_sub_epic is not None:
            current_story = {
                "idx": int(story_match.group(1)),
                "text": story_match.group(2).strip(),
                "tasks": [],
            }
            current_sub_epic["stories"].append(current_story)
            continue

        # Match "Task N:" (flat, no decimals like 1.1.1.1)
        task_match = re.match(r"^-?\s*Task\s+(\d+)\s*:\s*(.+)$", stripped, re.IGNORECASE)
        if task_match and current_story is not None:
            current_story["tasks"].append({
                "idx": int(task_match.group(1)),
                "text": task_match.group(2).strip(),
            })

    return epics


def _sanitize_backlog_response(response: str) -> str:
    text = _normalize_backlog_labels(response)
    cleaned_lines = []
    for line in text.splitlines():
        if re.match(r"(?im)^\s*(Status|Summary|Roles|Features|Timeline|Proposal\s*/\s*Input)\s*:", line):
            continue
        if re.match(r"(?im)^\s*(===\s*(Status|Summary|Roles|Features|Timeline|Proposal|Requirement).+===)\s*$", line):
            continue
        cleaned_lines.append(line)
    text = "\n".join(cleaned_lines).strip()

    # Backfill sub-epic lines when model emits Epic -> User Story directly (flat format).
    rebuilt = []
    epic_counter = 0
    sub_exists_for_epic = False
    for line in text.splitlines():
        stripped = line.strip()
        epic_match = re.match(r"^Epic\s*(\d+)\s*:\s*(.+)$", stripped, re.IGNORECASE)
        if epic_match:
            epic_counter = int(epic_match.group(1))
            sub_exists_for_epic = False
            rebuilt.append(f"Epic {epic_counter}: {epic_match.group(2).strip()}")
            continue

        # Match flat Sub-Epic (e.g., "Sub-Epic 1:", not "Sub-Epic 1.1:")
        if re.match(r"^-?\s*Sub-?Epic\s*\d+\s*:\s*.+$", stripped, re.IGNORECASE):
            sub_exists_for_epic = True
            rebuilt.append(stripped)
            continue

        # Match flat User Story (e.g., "User Story 1:", not "User Story 1.1.1:")
        if re.match(r"^-?\s*User\s+Story\s+\d+\s*:\s*.+$", stripped, re.IGNORECASE):
            if epic_counter > 0 and not sub_exists_for_epic:
                rebuilt.append(f"-Sub-Epic 1: Implementation for Epic {epic_counter}")
                sub_exists_for_epic = True
            rebuilt.append(stripped)
            continue

        rebuilt.append(stripped)

    text = "\n".join(rebuilt).strip()
    return text


def _has_malformed_headers(raw_text: str) -> bool:
    patterns = [
        r"(?im)^\s*Epi\s*$",
        r"(?im)^\s*C\s*\d+\s*:",
        r"(?im)^\s*-Sub-Ep(?!ic\s*1\s*:)",
        r"(?im)^\s*-Sub-\s*$",
        r"(?im)^\s*-User\s*$",
        r"(?im)^\s*-Task\s*$",
        r"(?im)^\s*UserStory\s*:",
    ]
    return any(re.search(p, raw_text) for p in patterns)


def _has_noise_markers(raw_text: str) -> bool:
    low = raw_text.lower()
    noise_tokens = [
        "-status",
        "-summary",
        "-roles",
        "-features",
        "-timeline",
        "-proposal",
        "-requirements",
        "end_of_backlog",
        "backlog hierarchy",
    ]
    return any(tok in low for tok in noise_tokens)


def _has_duplicate_headers(text: str) -> bool:
    """Check for duplicate headers in flat backlog format."""
    headers = []
    for line in text.splitlines():
        s = line.strip()
        if re.match(r"^Epic\s*\d+\s*:", s, re.IGNORECASE):
            headers.append(("epic", s.lower()))
        # Match flat Sub-Epic (e.g., "Sub-Epic 1:", not "Sub-Epic 1.1:")
        elif re.match(r"^-?\s*Sub-?Epic\s*\d+\s*:", s, re.IGNORECASE):
            headers.append(("sub_epic", s.lower()))
        # Match flat User Story (e.g., "User Story 1:", not "User Story 1.1.1:")
        elif re.match(r"^-?\s*User\s+Story\s+\d+\s*:", s, re.IGNORECASE):
            headers.append(("story", s.lower()))
        # Match flat Task (e.g., "Task 1:", not "Task 1.1.1.1:")
        elif re.match(r"^-?\s*Task\s+\d+\s*:", s, re.IGNORECASE):
            headers.append(("task", s.lower()))
    return len(headers) != len(set(headers))


def _is_generic_tasking(epics: list[dict]) -> bool:
    generic_phrases = [
        "plan and execute",
        "implement module",
        "develop feature",
        "complete development",
        "perform testing",
        "integrate backend",
    ]
    tasks = []
    for epic in epics:
        for sub in epic["sub_epics"]:
            for story in sub["stories"]:
                for task in story["tasks"]:
                    tasks.append(task["text"].strip().lower())

    if not tasks:
        return False

    generic_count = 0
    for t in tasks:
        if any(phrase in t for phrase in generic_phrases) and len(t) < 70:
            generic_count += 1

    return generic_count == len(tasks)


def _compliance_score(response: str) -> dict:
    epics = _parse_backlog_hierarchy(response)
    epic_count = len(epics)
    invalid_sub_epic_count = 0
    invalid_story_count = 0
    invalid_task_count = 0

    for epic in epics:
        if len(epic["sub_epics"]) != 1:
            invalid_sub_epic_count += 1
            continue
        sub = epic["sub_epics"][0]
        if len(sub["stories"]) != 1:
            invalid_story_count += 1
            continue
        story = sub["stories"][0]
        if len(story["tasks"]) != 2:
            invalid_task_count += 1

    has_forbidden = bool(
        re.search(
            r"(?im)^\s*(Status|Summary|Roles|Features|Timeline|Proposal\s*/\s*Input)\s*:",
            response,
        )
    )
    has_duplicates = _has_duplicate_headers(response)
    has_generic_tasks = _is_generic_tasking(epics)

    score = 0
    if 4 <= epic_count <= 6:
        score += 6
    score += max(0, 6 - abs(epic_count - 5))
    score += max(0, (epic_count - invalid_sub_epic_count)) * 2
    score += max(0, (epic_count - invalid_story_count)) * 2
    score += max(0, (epic_count - invalid_task_count)) * 3
    if not has_forbidden:
        score += 2
    if not has_duplicates:
        score += 2
    if not has_generic_tasks:
        score += 2

    return {
        "score": score,
        "epic_count": epic_count,
        "invalid_sub_epic_count": invalid_sub_epic_count,
        "invalid_story_count": invalid_story_count,
        "invalid_task_count": invalid_task_count,
        "has_forbidden": has_forbidden,
        "has_duplicates": has_duplicates,
        "has_generic_tasks": has_generic_tasks,
    }


def _generate(prompt_text: str, max_new_tokens: int = 420) -> str:
    inputs = tokenizer(prompt_text, return_tensors="pt")
    if torch.cuda.is_available():
        inputs = {k: v.cuda() for k, v in inputs.items()}

    with torch.inference_mode():
        outputs = model_infer.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
            do_sample=False,
            repetition_penalty=1.12,
            bad_words_ids=_BAD_IDS,
        )
    return tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()


def _extract_goal_epic_titles(proposal_text: str) -> list[str]:
    goals = []
    in_goals = False
    for raw in proposal_text.splitlines():
        line = raw.strip()
        if not line:
            if in_goals:
                break
            continue

        if re.match(r"(?i)^goals?\s*:\s*$", line):
            in_goals = True
            continue

        if in_goals and re.match(r"(?i)^(timeline|features|roles|summary|status)\s*:\s*", line):
            break

        if in_goals:
            bullet = re.sub(r"^[-*+\d.)\s]+", "", line).strip()
            if bullet:
                goals.append(bullet)

    return goals[:6]


def _build_epic_anchor_block(proposal_text: str) -> str:
    goals = _extract_goal_epic_titles(proposal_text)
    if not goals:
        return ""

    lines = ["Use these exact epic anchors from Goals (keep order):"]
    for idx, goal in enumerate(goals, start=1):
        lines.append(f"- Epic {idx}: {goal}")
    lines.append("Do not create epics outside these anchors.")
    return "\n".join(lines) + "\n\n"


examples_by_row = {}
max_row_seen = 0
with open(dataset_file, "r", encoding="utf-8") as f:
    for row_num, line in enumerate(f, start=1):
        max_row_seen = row_num
        if row_num not in selected_row_set:
            continue
        line = line.strip()
        if not line:
            continue
        examples_by_row[row_num] = json.loads(line)
        if len(examples_by_row) == len(selected_row_set):
            break

missing_rows = [row for row in selected_rows_sorted if row not in examples_by_row]
if missing_rows:
    raise ValueError(
        f"Selected row numbers out of range for dataset ({dataset_file}). "
        f"Requested rows: {selected_rows_sorted}, loaded through row {max_row_seen}, missing: {missing_rows}."
    )

examples = [examples_by_row[row] for row in selected_rows_sorted]

print("=" * 80)
print("MODEL 2 INFERENCE TEST - FLAT BACKLOG FORMAT")
print("=" * 80)
print(f"Quick mode: {QUICK_RUN_MODE}")
print(f"Strategies: {STRATEGIES_TO_RUN}")
print(f"Adapter path: {PROMOTED_ADAPTER_DIR}")
print(f"Dataset: {dataset_file}")
print(f"Selected dataset rows (1-based): {selected_rows_sorted}")
print(f"Examples to run: {len(examples)}")
print(f"Show full output: {SHOW_FULL_OUTPUT}")
print(f"Export txt: {EXPORT_OUTPUT_TXT}")
print("Few-shot examples loaded: 0")
print("Prompt-only strategy disabled: True")
print(f"Strict prompt integration active: {strict_prompt_integration_active}")
print(f"Strict prompt rule lines injected: {len(strict_rule_lines)}")
if USE_STRICT_PROMPT_RULES and not strict_prompt_integration_active:
    print("Strict backlog prompt rules unavailable; using base guided prefix.")
print("=" * 80)

pass_count = 0
export_rows = []
for idx, example in enumerate(examples, start=1):
    proposal = example["prompt"]
    expected_epic_count = len(_extract_goal_epic_titles(proposal)) or 5
    epic_anchor_block = _build_epic_anchor_block(proposal)
    candidates = []

    if "B" in STRATEGIES_TO_RUN:
        prompt_b = GUIDED_PREFIX + epic_anchor_block + "Input:\n" + proposal + "\n\nBacklog:\n"
        raw_b = _generate(prompt_b, max_new_tokens=TOKENS_BY_STRATEGY["B"])
        clean_b = _sanitize_backlog_response(raw_b)
        score_b = _compliance_score(clean_b)
        needs_retry_b = (
            score_b["epic_count"] != expected_epic_count
            or score_b["invalid_sub_epic_count"] > 0
            or score_b["invalid_story_count"] > 0
            or score_b["invalid_task_count"] > 0
            or _has_malformed_headers(raw_b)
            or _has_noise_markers(raw_b)
            or not clean_b.lstrip().lower().startswith("epic 1:")
            or OUTPUT_END_MARKER not in raw_b
        )
        if needs_retry_b:
            retry_b = (
                GUIDED_PREFIX
                + epic_anchor_block
                + "Input:\n" + proposal + "\n\n"
                + "Return only flat backlog. Start with Epic 1:. Use exactly one Sub-Epic 1, one User Story 1, and Task 1/Task 2 per Epic. "
                + f"Generate exactly {expected_epic_count} epics. End with {OUTPUT_END_MARKER}.\n\nBacklog:\n"
            )
            raw_b = _generate(retry_b, max_new_tokens=min(TOKENS_BY_STRATEGY["B"] + 100, 620))
            clean_b = _sanitize_backlog_response(raw_b)
            score_b = _compliance_score(clean_b)
        candidates.append(("B: compact strict", clean_b, score_b))

    if "C" in STRATEGIES_TO_RUN:
        prompt_c = GUIDED_PREFIX + epic_anchor_block + "Input:\n" + proposal + "\n\nBacklog:\n"
        raw_c = _generate(prompt_c, max_new_tokens=TOKENS_BY_STRATEGY["C"])
        clean_c = _sanitize_backlog_response(raw_c)
        score_c = _compliance_score(clean_c)
        needs_retry_c = (
            score_c["epic_count"] != expected_epic_count
            or score_c["invalid_sub_epic_count"] > 0
            or score_c["invalid_story_count"] > 0
            or score_c["invalid_task_count"] > 0
            or _has_malformed_headers(raw_c)
            or _has_noise_markers(raw_c)
            or not clean_c.lstrip().lower().startswith("epic 1:")
            or OUTPUT_END_MARKER not in raw_c
        )
        if needs_retry_c:
            retry_c = (
                GUIDED_PREFIX
                + epic_anchor_block
                + "Input:\n" + proposal + "\n\n"
                + "Return only flat backlog. Start with Epic 1:. Use exact hierarchy headers and do not add extra sections. "
                + f"Generate exactly {expected_epic_count} epics and end with {OUTPUT_END_MARKER}.\n\nBacklog:\n"
            )
            raw_c = _generate(retry_c, max_new_tokens=min(TOKENS_BY_STRATEGY["C"] + 100, 620))
            clean_c = _sanitize_backlog_response(raw_c)
            score_c = _compliance_score(clean_c)
        candidates.append(("C: compact strict+anchor", clean_c, score_c))

    if not candidates:
        raise RuntimeError("No inference strategies selected. Set STRATEGIES_TO_RUN to B and/or C.")

    best_name, best_out, best_score = max(candidates, key=lambda x: x[2]["score"])

    compliant = (
        best_score["epic_count"] == expected_epic_count
        and best_score["invalid_sub_epic_count"] == 0
        and best_score["invalid_story_count"] == 0
        and best_score["invalid_task_count"] == 0
        and not best_score["has_forbidden"]
        and not best_score["has_duplicates"]
        and not best_score["has_generic_tasks"]
    )

    print(f"\n{'=' * 80}")
    print(f"EXAMPLE {idx} (dataset row {selected_rows_sorted[idx - 1]})")
    print(f"Best strategy: {best_name} (score={best_score['score']})")
    print(
        f"Epics: {best_score['epic_count']} | "
        f"Invalid sub-epic blocks: {best_score['invalid_sub_epic_count']} | "
        f"Invalid story blocks: {best_score['invalid_story_count']} | "
        f"Invalid task blocks: {best_score['invalid_task_count']} | "
        f"Forbidden: {best_score['has_forbidden']} | "
        f"Duplicates: {best_score['has_duplicates']} | "
        f"Generic tasks: {best_score['has_generic_tasks']}"
    )
    print(f"Result: {'PASS' if compliant else 'FAIL'}")
    print("Output:")
    if SHOW_FULL_OUTPUT:
        print(best_out)
    else:
        print(best_out[:400])

    export_rows.append({
        "example": idx,
        "dataset_row": selected_rows_sorted[idx - 1],
        "best_strategy": best_name,
        "score": best_score["score"],
        "compliant": compliant,
        "epic_count": best_score["epic_count"],
        "invalid_sub_epic_count": best_score["invalid_sub_epic_count"],
        "invalid_story_count": best_score["invalid_story_count"],
        "invalid_task_count": best_score["invalid_task_count"],
        "has_forbidden": best_score["has_forbidden"],
        "has_duplicates": best_score["has_duplicates"],
        "has_generic_tasks": best_score["has_generic_tasks"],
        "proposal": proposal,
        "output": best_out,
    })

    if compliant:
        pass_count += 1

print(f"\n{'=' * 80}")
print(f"COMPLIANCE SUMMARY: {pass_count}/{len(examples)} fully compliant")
print("=" * 80)

if EXPORT_OUTPUT_TXT:
    EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    export_file = EXPORT_DIR / f"model2_inference_outputs_{stamp}.txt"

    lines = []
    lines.append("MODEL 2 INFERENCE TEST EXPORT")
    lines.append(f"Generated: {datetime.now().isoformat(timespec='seconds')}")
    lines.append(f"Dataset: {dataset_file}")
    lines.append(f"Selected rows (1-based): {selected_rows_sorted}")
    lines.append(f"Strict prompt integration active: {strict_prompt_integration_active}")
    lines.append(f"Strict prompt rule lines injected: {len(strict_rule_lines)}")
    lines.append(f"Examples run: {len(examples)}")
    lines.append(f"Compliance summary: {pass_count}/{len(examples)}")
    lines.append("=" * 100)

    for row in export_rows:
        lines.append("")
        lines.append("-" * 100)
        lines.append(f"EXAMPLE {row['example']} (dataset row {row['dataset_row']})")
        lines.append(f"Best strategy: {row['best_strategy']} | Score: {row['score']} | Compliant: {row['compliant']}")
        lines.append(
            "Checks: "
            f"epics={row['epic_count']}, "
            f"invalid_sub_epic={row['invalid_sub_epic_count']}, "
            f"invalid_story={row['invalid_story_count']}, "
            f"invalid_task={row['invalid_task_count']}, "
            f"forbidden={row['has_forbidden']}, "
            f"duplicates={row['has_duplicates']}, "
            f"generic_tasks={row['has_generic_tasks']}"
        )
        lines.append("")
        lines.append("Proposal / Input:")
        lines.append(row["proposal"].strip())
        lines.append("")
        lines.append("Model Output:")
        lines.append(row["output"].strip())

    export_file.write_text("\n".join(lines), encoding="utf-8")
    print(f"\nSaved inference outputs to: {export_file}")



Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

MODEL 2 INFERENCE TEST - FLAT BACKLOG FORMAT
Quick mode: True
Strategies: ['B']
Adapter path: C:\Users\Aaron\GitHub Repos\My-Crew-Manager\AI\llms\fine_tune\qwen_model2_backlog_lora_1p5b
Dataset: c:\Users\Aaron\GitHub Repos\My-Crew-Manager\AI\llms\fine_tune\dataset\model2_part1_to_backlog_epic_v1.jsonl
Selected dataset rows (1-based): [7]
Examples to run: 1
Show full output: True
Export txt: True
Few-shot examples loaded: 0
Prompt-only strategy disabled: True
Strict prompt integration active: True
Strict prompt rule lines injected: 7

EXAMPLE 1 (dataset row 7)
Best strategy: B: compact strict (score=53)
Epics: 5 | Invalid sub-epic blocks: 0 | Invalid story blocks: 0 | Invalid task blocks: 0 | Forbidden: False | Duplicates: False | Generic tasks: False
Result: PASS
Output:
Epic 1: Design lesson module
-Sub-Epic 1: Week 1 setup
-User Story 1: As a Project Manager, I want to set up the mobile app framework for the first week of development so that we can start building the lesson module ne

## Step 6.1: Compare dataset target vs generated output

Use the same fixed inference rows to compare ground-truth dataset responses against fresh model outputs.

In [20]:
# Compare expected dataset responses vs fresh model outputs on selected rows.
# Requires Step 6 helper functions to be available in kernel state.

required_names = [
    "selected_rows_sorted",
    "examples_by_row",
    "_sanitize_backlog_response",
    "_compliance_score",
    "_generate",
    "GUIDED_PREFIX",
]
missing = [name for name in required_names if name not in globals()]
if missing:
    raise RuntimeError(
        "Run Step 6 first so comparison helpers are loaded. Missing: " + ", ".join(missing)
    )

print("=" * 90)
print("MODEL 2 DATASET VS GENERATED COMPARISON")
print("=" * 90)

comparison_rows = selected_rows_sorted
comparison_results = []

for row_num in comparison_rows:
    example = examples_by_row[row_num]
    proposal = example["prompt"]
    target_raw = example.get("response", "")
    target_clean = _sanitize_backlog_response(target_raw)
    target_score = _compliance_score(target_clean)

    epic_anchor_block = _build_epic_anchor_block(proposal) if "_build_epic_anchor_block" in globals() else ""
    prompt_cmp = GUIDED_PREFIX + epic_anchor_block + "Input:\n" + proposal + "\n\nBacklog:\n"
    generated_raw = _generate(prompt_cmp, max_new_tokens=TOKENS_BY_STRATEGY.get("B", 520))
    generated_clean = _sanitize_backlog_response(generated_raw)
    generated_score = _compliance_score(generated_clean)

    result = {
        "row": row_num,
        "target_score": target_score,
        "generated_score": generated_score,
        "target_output": target_clean,
        "generated_output": generated_clean,
    }
    comparison_results.append(result)

    print(f"\nRow {row_num}")
    print(
        "Target   -> "
        f"score={target_score['score']}, epics={target_score['epic_count']}, "
        f"invalid_sub={target_score['invalid_sub_epic_count']}, invalid_story={target_score['invalid_story_count']}, "
        f"invalid_task={target_score['invalid_task_count']}, forbidden={target_score['has_forbidden']}, "
        f"duplicates={target_score['has_duplicates']}, generic={target_score['has_generic_tasks']}"
    )
    print(
        "Generated-> "
        f"score={generated_score['score']}, epics={generated_score['epic_count']}, "
        f"invalid_sub={generated_score['invalid_sub_epic_count']}, invalid_story={generated_score['invalid_story_count']}, "
        f"invalid_task={generated_score['invalid_task_count']}, forbidden={generated_score['has_forbidden']}, "
        f"duplicates={generated_score['has_duplicates']}, generic={generated_score['has_generic_tasks']}"
    )

target_pass = sum(
    1 for r in comparison_results
    if 4 <= r["target_score"]["epic_count"] <= 6
    and r["target_score"]["invalid_sub_epic_count"] == 0
    and r["target_score"]["invalid_story_count"] == 0
    and r["target_score"]["invalid_task_count"] == 0
    and not r["target_score"]["has_forbidden"]
    and not r["target_score"]["has_duplicates"]
    and not r["target_score"]["has_generic_tasks"]
)
generated_pass = sum(
    1 for r in comparison_results
    if 4 <= r["generated_score"]["epic_count"] <= 6
    and r["generated_score"]["invalid_sub_epic_count"] == 0
    and r["generated_score"]["invalid_story_count"] == 0
    and r["generated_score"]["invalid_task_count"] == 0
    and not r["generated_score"]["has_forbidden"]
    and not r["generated_score"]["has_duplicates"]
    and not r["generated_score"]["has_generic_tasks"]
)

print("\n" + "=" * 90)
print(f"Target compliant rows: {target_pass}/{len(comparison_results)}")
print(f"Generated compliant rows: {generated_pass}/{len(comparison_results)}")



MODEL 2 DATASET VS GENERATED COMPARISON


KeyboardInterrupt: 

## Step 6.2 (Conditional): Convert Dataset to Flat Format

If compliance < 80% (0/5 or 1/5 rows pass), regenerate training data with flat numbering (no decimals).


In [8]:
import re
import json

# Conversion needed: 0/5 compliance indicates distribution shift (trained on decimal, inferring flat)
# Regenerate training dataset with flat format (no decimals: Epic 1, Sub-Epic 1, User Story 1, Task 1-2 per epic)

def convert_decimal_to_flat(response_text: str) -> str:
    """Convert backlog from decimal notation (1.1.1.1) to flat notation (1, 1, 1, 1-2)."""
    lines = []
    
    for raw_line in response_text.splitlines():
        line = raw_line.strip()
        if not line:
            lines.append("")
            continue
        
        # Match Epic X: with decimal or not
        epic_match = re.match(r"(?i)^(epic\s+)(\d+)(\.\d+)?:\s*(.+)$", line)
        if epic_match:
            # Extract just the epic number, drop any decimal suffix
            epic_num = epic_match.group(2)
            title = epic_match.group(4)
            # Remove the *(covers: ...)* suffix if present
            title = re.sub(r"\s*\*\(covers:\s*.+?\)\*\s*$", "", title)
            lines.append(f"Epic {epic_num}: {title}")
            continue
        
        # Match Sub-Epic X.Y: → Sub-Epic X:
        sub_match = re.match(r"(?i)^(-?\s*sub-?epic\s+)(\d+)(\.\d+)+:\s*(.+)$", line)
        if sub_match:
            epic_num = sub_match.group(2)
            title = sub_match.group(4)
            lines.append(f"-Sub-Epic {epic_num}: {title}")
            continue
        
        # Match User Story X.Y.Z: → User Story X:
        story_match = re.match(r"(?i)^(-?\s*user\s+story\s+)(\d+)(\.\d+)*:\s*(.+)$", line)
        if story_match:
            epic_num = story_match.group(2)
            text = story_match.group(4)
            lines.append(f"-User Story {epic_num}: {text}")
            continue
        
        # Match Task X.Y.Z.W: → Task 1: or Task 2: (depending on which task it is)
        task_match = re.match(r"(?i)^(-?\s*task\s+)(\d+)(\.\d+)*:\s*(.+)$", line)
        if task_match:
            full_num = task_match.group(2)
            # Extract the last digit (should be 1 or 2 for current format X.X.X.1 or X.X.X.2)
            parts = full_num.split('.')
            last_digit = parts[-1]
            text = task_match.group(4)
            lines.append(f"-Task {last_digit}: {text}")
            continue
        
        # Keep other lines as-is
        lines.append(line)
    
    return "\n".join(lines)


# Test conversion on first two rows
if "dataset_file" not in globals():
    dataset_file = DATASET_DIR / "model2_part1_to_backlog_epic_v1.jsonl"

test_rows = []
print("DATASET FLAT FORMAT CONVERSION - SAMPLE TEST")
print("=" * 80)

with open(dataset_file, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= 2:
            break
        row = json.loads(line.strip())
        test_rows.append(row)
        
        original_response = row["response"]
        converted_response = convert_decimal_to_flat(original_response)
        
        print(f"\nRow {i+1} ORIGINAL (decimal format):")
        print(original_response[:300] + "..." if len(original_response) > 300 else original_response)
        print(f"\nRow {i+1} CONVERTED (flat format):")
        print(converted_response[:300] + "..." if len(converted_response) > 300 else converted_response)
        print("-" * 80)

# Now regenerate the entire dataset in flat format
print("\nRegenerating entire dataset in flat format...")
input_file = dataset_file
output_file = dataset_file.parent / "model2_part1_to_backlog_epic_v1_flat.jsonl"

total_converted = 0
with open(input_file, "r", encoding="utf-8") as infile, \
     open(output_file, "w", encoding="utf-8") as outfile:
    for line in infile:
        if not line.strip():
            continue
        row = json.loads(line)
        row["response"] = convert_decimal_to_flat(row["response"])
        outfile.write(json.dumps(row, ensure_ascii=False) + "\n")
        total_converted += 1

print(f"Converted {total_converted} examples")
print(f"Saved to: {output_file}")

# Now replace the original dataset with the flat version
import shutil
shutil.copy2(output_file, input_file)
print(f"Backup original to: {input_file}.backup")
shutil.copy2(input_file, str(input_file) + ".backup")
print(f"Replaced {input_file} with flat format version")

DATASET FLAT FORMAT CONVERSION - SAMPLE TEST

Row 1 ORIGINAL (decimal format):
Epic 1: Design venue search interface *(covers: Design venue search interface)*
 -Sub-Epic 1.1: Implement Venue Search workflow
  -User Story 1.1.1: As a user, I want to search venues
   -Task 1.1.1.1: Build venue search API
   -Task 1.1.1.2: Create venue search UI
Epic 2: Implement vendor managemen...

Row 1 CONVERTED (flat format):
Epic 1: Design venue search interface
-Sub-Epic 1: Implement Venue Search workflow
-User Story 1: As a user, I want to search venues
-Task 1: Build venue search API
-Task 1: Create venue search UI
Epic 2: Implement vendor management system
-Sub-Epic 2: Implement Vendor Management workflow
-User Stor...
--------------------------------------------------------------------------------

Row 2 ORIGINAL (decimal format):
Epic 1: Design diet tracking interface *(covers: Design diet tracking interface)*
 -Sub-Epic 1.1: Implement Diet Tracking workflow
  -User Story 1.1.1: As a pet owner

In [10]:
import shutil

# Clear tokenized dataset cache since we changed the source data format
tokenized_path = TOKENIZED_DIR / "tokenized_model2_qwen_epic_dualprompt_v3"
if tokenized_path.exists():
    shutil.rmtree(tokenized_path)
    print(f"Cleared tokenized cache: {tokenized_path}")

Cleared tokenized cache: c:\Users\Aaron\GitHub Repos\My-Crew-Manager\AI\llms\fine_tune\tokenized\tokenized_model2_qwen_epic_dualprompt_v3
